# SAE Training — CLIP / SigLIP1 / SigLIP2

Trains Sparse Autoencoders on frozen ViT activations for all 12 layers.
Uses **topk** activation: `k` directly sets L0 (active features per patch).
Metrics logged to Weights & Biases.

Run cells top to bottom. Runtime check -> config -> wandb login -> training.


## 1 — Runtime check


In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')


Fri Jul  3 00:29:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2 — Download dataset from Drive


In [2]:
import os, glob

FOLDER_ID    = '16axcxIfAwW0ebpc_0NuJSeSy_RDY_zqt'   # Drive folder with the val parquets
SAVE_DIR     = '/content/saes'
DATA_DIR     = '/content/imagenet_val'
PARQUET_GLOB = '/content/imagenet_val/data/*.parquet'

# Uncomment to force a clean re-download if a previous pull was partial:
# !rm -rf /content/imagenet_val
!pip install -q gdown
!gdown --folder "https://drive.google.com/drive/folders/{FOLDER_ID}" -O /content --quiet

print(f'{len(glob.glob(PARQUET_GLOB))} parquet files found')


14 parquet files found


## 3 — Install dependencies
ViT-Prisma is installed from our fork (adds SigLIP1/2 support).


In [3]:
!pip install -q transformers==4.44.2 einops timm datasets huggingface_hub tqdm wandb
!pip install -q git+https://github.com/asharalam11/ViT-Prisma.git@add_siglip2


  Preparing metadata (setup.py) ... done


## 4 — Weights & Biases login
Paste your API key from wandb.ai/authorize when prompted.


In [4]:
import wandb
wandb.login()


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: asharalam to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 5 — Load frozen model + data (once, reused for every layer)


In [5]:
import torch
from torchvision import transforms
from datasets import load_dataset
from vit_prisma.models.model_loader import load_hooked_model
from vit_prisma.sae import VisionModelSAERunnerConfig, VisionSAETrainer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

# --- config knobs -------------------------------------------------
MODEL_ID    = 'google/siglip-base-patch16-224'   # or siglip2 / clip
HOOK_POINT  = 'hook_resid_post'                   # or hook_mlp_out
K           = 32                                  # topk: active features per patch == target L0
PASSES      = 3                                   # full passes over the dataset
LR          = 1e-4
TAG         = f'topk_{K}'                          # tags Drive folder + checkpoint filenames

# A100 (40GB) batch sizes — crux test used 0.57GB at store_batch 8, so lots of headroom
TRAIN_BATCH    = 16384    # SAE optimizer batch (tokens); larger = fewer steps, faster
STORE_BATCH    = 128      # images per ViT forward when harvesting activations
BUFFER_BATCHES = 20       # activation buffer depth (~1.5GB at these sizes)

# wandb
WANDB_PROJECT = 'siglip-sae'
WANDB_ENTITY  = None      # None = your default wandb entity
# ------------------------------------------------------------------

# The activations store expects each item as (image_tensor, label).
# Our parquet rows are {'image': PIL, 'label': int}, so wrap them.
_preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

class ImageTupleDataset(torch.utils.data.Dataset):
    def __init__(self, hf_ds):
        self.ds = hf_ds
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        row = self.ds[i]
        return _preprocess(row['image'].convert('RGB')), row['label']

full  = load_dataset('parquet', data_files=PARQUET_GLOB, split='train')
split = full.train_test_split(test_size=0.02, seed=42)
train_ds = ImageTupleDataset(split['train'])
eval_ds  = ImageTupleDataset(split['test'])
print(f'train: {len(train_ds)}  eval: {len(eval_ds)}')

model = load_hooked_model(MODEL_ID, device=device).to(device)
model.eval()
print('Model loaded on', device)


/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




Using device: cuda


Generating train split: 0 examples [00:00, ? examples/s]

train: 49000  eval: 1000


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning:


Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).



config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

ln_pre not set


model.safetensors:   0%|          | 0.00/813M [00:00<?, ?B/s]

Model loaded on cuda


## 6 — Drive upload helper (creates a per-run subfolder)


In [6]:
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

PARENT_ID = FOLDER_ID
_creds, _ = google.auth.default()
_drive = build('drive', 'v3', credentials=_creds)

FOLDER_NAME = f'sae_siglip_ckpt_{TAG}'
q = (f"name='{FOLDER_NAME}' and '{PARENT_ID}' in parents "
     "and mimeType='application/vnd.google-apps.folder' and trashed=false")
hits = _drive.files().list(q=q, fields='files(id)').execute()['files']
if hits:
    SAE_FOLDER_ID = hits[0]['id']
else:
    meta = {'name': FOLDER_NAME, 'parents': [PARENT_ID],
            'mimeType': 'application/vnd.google-apps.folder'}
    SAE_FOLDER_ID = _drive.files().create(body=meta, fields='id').execute()['id']
print(FOLDER_NAME, 'folder id:', SAE_FOLDER_ID)

def upload_to_drive(path):
    meta  = {'name': os.path.basename(path), 'parents': [SAE_FOLDER_ID]}
    media = MediaFileUpload(path, resumable=True)
    _drive.files().create(body=meta, media_body=media, fields='id').execute()
    print('Uploaded', os.path.basename(path))


sae_siglip_ckpt_topk_32 folder id: 1FbaobdFHNB-Ipv2IAP2Cah-Oy4MGbuAW


## 7 — Train all 12 layers

`k` fixes L0 directly, so there is no L1 to tune. Watch **MSE** per layer in wandb:
low + flat = k is enough; rising at deep layers = bump `k` to 64.
Each layer is a separate wandb run named `{model}_{hook}_topk{k}_layer{N}`.


In [11]:
def make_cfg(layer, k, passes):
    cfg = VisionModelSAERunnerConfig(
        model_name=MODEL_ID,
        model_class_name='HookedViT',
        hook_point_layer=layer,
        layer_subtype=HOOK_POINT,
        d_in=768,
        expansion_factor=4,
        context_size=196,                # 14x14 patches, no CLS for SigLIP
        image_size=224,
        activation_fn_str='topk',
        activation_fn_kwargs={'k': k},
        l1_coefficient=1e-8,             # topk enforces sparsity; tiny nonzero avoids /0 in wandb logging
        lr=LR,
        train_batch_size=TRAIN_BATCH,
        store_batch_size=STORE_BATCH,
        n_batches_in_buffer=BUFFER_BATCHES,
        n_checkpoints=0,
        log_to_wandb=True,
        wandb_project=WANDB_PROJECT,
        wandb_entity=WANDB_ENTITY,
        wandb_log_frequency=10,
        checkpoint_path=SAVE_DIR,
        verbose=False,
        _device=device,
    )
    # library hardcodes dataset_size=1.3M; scale num_epochs so we get `passes` real passes
    cfg.num_epochs = passes * len(train_ds) / 1_300_000
    return cfg

import os, re

# --- resume guard: skip any layer already uploaded to Drive ---
existing = _drive.files().list(
    q=f"'{SAE_FOLDER_ID}' in parents and trashed=false",
    fields='files(name)').execute()['files']
done_layers = {int(m.group(1)) for f in existing
               if (m := re.search(r'layer(\d+)\.pt$', f['name']))}
print('already in Drive, will skip:', sorted(done_layers))

# make a wandb connection hiccup wait/retry instead of killing the process
os.environ.setdefault("WANDB__SERVICE_WAIT", "300")

os.makedirs(SAVE_DIR, exist_ok=True)
for LAYER in range(12):
    if LAYER in done_layers:
        print(f'skip layer {LAYER} (already done)')
        continue
    cfg = make_cfg(LAYER, K, PASSES)
    trainer = VisionSAETrainer(cfg, model, train_ds, eval_dataset=eval_ds)
    trainer.cfg.run_name = f'{MODEL_ID.split("/")[-1]}_{HOOK_POINT}_{TAG}_layer{LAYER}'
    sae = trainer.run()
    save_path = f'{SAVE_DIR}/sae_{MODEL_ID.split("/")[-1]}_{HOOK_POINT}_{TAG}_layer{LAYER}.pt'
    torch.save(sae.state_dict(), save_path)
    upload_to_drive(save_path)
    wandb.finish()
    print(f'=== layer {LAYER} done, L0/MSE in wandb ===')


Not saving checkpoints so skipping creating checkpoint directory


Objective value: 3128019.0000:   2%|▏         | 4/200 [00:00<00:02, 84.18it/s]
Training SAE: Loss: 0.0036, MSE Loss: 0.0036, L1 Loss: 0.0000, L0: 32.0000: : 28819456it [05:45, 83349.81it/s]                             


Uploaded sae_siglip-base-patch16-224_hook_resid_post_topk_32_layer0.pt


details/current_learning_rate,▁▂▂▃▃▄▅▅▆████████▇▇▇▆▆▆▆▅▅▃▃▃▃▂▂▂▁▁▁▁▁▁▁
details/n_training_images,▁▁▁▁▁▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▅▅▆▇▇▇▇▇▇▇██
details/n_training_tokens,▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇▇█████
losses/ghost_grad_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
losses/l1_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
losses/mse_loss,██▇▆▆▅▅▄▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
losses/overall_loss,██▇▅▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
metrics/explained_variance,▁▁▂▃▃▄▅▅▆▆▆▆▇▇▇█████████████████████████
metrics/explained_variance_std,█▇▇▄▂▁▂▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
metrics/l0,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+5,...


=== layer 0 done, L0/MSE in wandb ===
Not saving checkpoints so skipping creating checkpoint directory


...

Uploaded sae_siglip-base-patch16-224_hook_resid_post_topk_32_layer3.pt


details/current_learning_rate,▂▂▃▃▃▅▅▆▆▇█████▇▇▇▇▇▆▅▅▅▄▄▄▃▃▃▃▂▂▂▁▁▁▁▁▁
details/n_training_images,▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇█████
details/n_training_tokens,▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇████
losses/ghost_grad_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
losses/l1_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
losses/mse_loss,██▇▆▆▄▄▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
losses/overall_loss,█████▆▅▅▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
metrics/explained_variance,▁▁▁▂▂▃▄▄▅▆▆▇▇▇▇█████████████████████████
metrics/explained_variance_std,▃▃▂▂▁▆▇▇██▇█▇▇▇▆▆▆▆▆▆▆▆▅▆▅▅▆▅▆▆▅▆▆▅▅▅▅▆▆
metrics/l0,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+5,...


=== layer 3 done, L0/MSE in wandb ===
Not saving checkpoints so skipping creating checkpoint directory


Objective value: 2939094.0000:   1%|          | 2/200 [00:00<00:02, 67.48it/s]
Training SAE: Loss: 0.0043, MSE Loss: 0.0043, L1 Loss: 0.0000, L0: 32.0000:  49%|████▉     | 14123008/28812000 [04:18<03:22, 72480.95it/s]

: 

## 8 — Optional: single-layer smoke test
Run this **instead of** cell 7 to verify the pipeline on one layer (~1 min at 0.05 passes) before the full run.


In [10]:
LAYER = 8
cfg = make_cfg(LAYER, K, 0.05)          # tiny fraction of a pass
trainer = VisionSAETrainer(cfg, model, train_ds, eval_dataset=eval_ds)
trainer.cfg.run_name = f'smoketest_layer{LAYER}'
sae = trainer.run()
wandb.finish()
print('smoke test done — L0 should be ~', K)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning:

This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning:

This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.



Not saving checkpoints so skipping creating checkpoint directory






Objective value: 5449112.0000:   2%|▏         | 3/200 [00:00<00:02, 68.26it/s]

Training SAE: Loss: 0.0150, MSE Loss: 0.0150, L1 Loss: 0.0000, L0: 32.0000:   1%|          | 147456/28812000 [07:04<22:56:25, 347.09it/s]

Training SAE: Loss: 0.0203, MSE Loss: 0.0203, L1 Loss: 0.0000, L0: 32.0000: : 491520it [01:26, 5699.91it/s]                          


details/current_learning_rate,▁▄█
details/n_training_images,▁▅█
details/n_training_tokens,▁▅█
losses/ghost_grad_loss,▁▁▁
losses/l1_loss,▁▁▁
losses/mse_loss,█▁▂
losses/overall_loss,█▁▂
metrics/explained_variance,█▁▅
metrics/explained_variance_std,█▁▆
metrics/l0,▁▁▁
+2,...


smoke test done — L0 should be ~ 32
